<a href="https://colab.research.google.com/github/akshita-singh-2808/airlines_churn_prediction/blob/main/airlines_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✈️ Airline Customer Churn Prediction: A Data-Driven Approach

This notebook presents a comprehensive analysis of airline customer data to identify key factors contributing to churn and develop actionable strategies for retention. It encompasses data cleaning, exploratory data analysis, feature engineering, and churn labeling, culminating in a robust dataset ready for machine learning model development.

##  Business Problem

Customer churn is a critical challenge for airlines, directly impacting revenue and long-term profitability. Retaining existing customers is often more cost-effective than acquiring new ones. This project aims to:

1.  **Identify drivers of churn**: Understand what demographic, behavioral, and loyalty-related factors cause customers to leave.
2.  **Predict churn**: Develop a robust dataset that can be used to build a predictive model to identify customers at high risk of churning.
3.  **Inform retention strategies**: Provide data-driven insights and recommendations for targeted interventions to reduce churn and enhance customer loyalty.

## Setup and Data Loading

This section handles the initial environment setup, including mounting Google Drive for data access and loading the necessary datasets into pandas DataFrames. We are working with four distinct datasets:

*   `Airline Loyalty Data Dictionary.csv`: Provides metadata and descriptions for the other datasets.
*   `Customer Flight Activity.csv`: Contains detailed records of customer flight behavior.
*   `Customer Loyalty History.csv`: Stores historical information about customer loyalty programs.
*   `Calendar.csv`: Likely contains time-based information to support date-related analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
dictionary= pd.read_csv("Airline Loyalty Data Dictionary.csv")
activity = pd.read_csv("Customer Flight Activity.csv")
loyalty = pd.read_csv("Customer Loyalty History.csv")
calendar = pd.read_csv("Calendar.csv")

## Data Cleaning


In [ ]:
loyalty

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16732,823768,Canada,British Columbia,Vancouver,V6E 3Z3,Female,College,NaN,Married,Star,61850.19,Standard,2012,12,NaN,NaN
16733,680886,Canada,Saskatchewan,Regina,S1J 3C5,Female,Bachelor,89210.0,Married,Star,67907.27,Standard,2014,9,NaN,NaN
16734,776187,Canada,British Columbia,Vancouver,V5R 1W3,Male,College,NaN,Single,Star,74228.52,Standard,2014,3,NaN,NaN
16735,906428,Canada,Yukon,Whitehorse,Y2K 6R0,Male,Bachelor,-57297.0,Married,Star,10018.66,2018 Promotion,2018,4,NaN,NaN


###  Initial Data Inspection

Before cleaning, it's good practice to inspect the data types and identify missing values. This helps us understand the scope of cleaning required.

In [ ]:
loyalty.isnull().sum()

,0
Loyalty Number,0
Country,0
Province,0
City,0
Postal Code,0
Gender,0
Education,0
Salary,4238
Marital Status,0
Loyalty Card,0


# STEP 1 : FIX BROKEN SALARIES
The problem — 4,258 records with broken salary

4,238 rows had NaN (blank salary)
20 rows had negative salary (e.g. –$58,486) — impossible, clearly a data entry mistake (someone probably typed without the dollar sign, or it was a system glitch flipping the sign)

In [ ]:
loyalty.loc[loyalty['Salary'] < 0, 'Salary'] = np.nan   # negatives → blank


#### Marking Imputed Salaries

We create a boolean flag, `Salary_Was_Imputed`, to indicate records where the original salary was either missing or invalid. This can be a useful feature for our model, as imputed values might carry different predictive weight.

In [ ]:
loyalty['Salary_Was_Imputed'] = loyalty['Salary'].isnull()    #if the salary is null it will show true else false

#### Imputing Missing Salaries

To fill the `NaN` values, we use a more sophisticated imputation strategy. Salaries are imputed based on the median salary of customers with the same `Education` level and `Loyalty Card` type. This approach assumes that salary distributions vary by education and loyalty tier, making the imputation more accurate than a simple overall median.

In [ ]:
loyalty['Salary_Imputed'] = loyalty.groupby(     #calculate the median salary using loyalty card and education as group
    ['Education', 'Loyalty Card']
)['Salary'].transform(
    lambda x: x.fillna(x.median())
)


#### Handling Remaining Missing Salaries

In cases where a specific `Education` and `Loyalty Card` combination might still result in `NaN` (e.g., if an entire group has missing salaries), we apply a fallback: fill any remaining `NaN` values with the overall median salary from the original, non-negative `Salary` column. This ensures all `Salary_Imputed` values are complete.

In [ ]:
loyalty['Salary_Imputed'] = loyalty['Salary_Imputed'].fillna(
    loyalty['Salary'].median()
)

 #  Step 2: Removing Ambiguous Cancellation Records
 DROP 11 BAD CANCELLATION RECORDS

11 customer records were identified where the cancellation month was the same as or earlier than the enrollment month. Since the dataset only contains month-level granularity (not exact dates), these records are considered ambiguous for churn labeling. For example, a customer could realistically enroll on 10-Dec-2017 and cancel on 25-Dec-2017, which would still appear as the same month in the dataset. However, because exact dates are unavailable, it is impossible to distinguish valid same-month cancellations from logically inconsistent cases. To maintain cleaner and more reliable churn labels for modeling, these 11 records were excluded from the training dataset.

In [ ]:
loyalty['enroll_ym'] = loyalty['Enrollment Year'] * 100 + loyalty['Enrollment Month']  #this is done so that dates can be comparable

In [ ]:
cancelled = loyalty[loyalty['Cancellation Year'].notna()].copy()  #this keeps only cancelled customers

In [ ]:
cancelled['cancel_ym'] = cancelled['Cancellation Year'] * 100 + cancelled['Cancellation Month']

In [ ]:
cancelled['is_bad'] = cancelled['cancel_ym'] <= cancelled['enroll_ym']  #impossible conditions - when the cancellation is done before enrollment


In [ ]:
print(cancelled[cancelled['is_bad']][['Loyalty Number','Enrollment Year',
      'Enrollment Month','Cancellation Year','Cancellation Month']])


       Loyalty Number  Enrollment Year  Enrollment Month  Cancellation Year  \
1461           607266             2016                 3             2016.0   
1756           939593             2013                 9             2013.0   
1957           140042             2014                 8             2014.0   
3145           488724             2013                 1             2013.0   
3157           160779             2017                12             2017.0   
5732           472283             2014                 4             2014.0   
5804           239221             2014                 5             2014.0   
8860           896685             2016                12             2016.0   
10755          304528             2018                 8             2018.0   
12147          871455             2017                 6             2017.0   
13498          110603             2017                10             2017.0   

       Cancellation Month  
1461                  3

#### Final Cleaning: Removing Bad Records

Finally, the identified 'bad' cancellation records are removed from the `loyalty` DataFrame to create `loyalty_clean`. This ensures our churn analysis is based on a logically consistent dataset.

In [ ]:
# Remove bad customers
loyalty_clean = loyalty[
    ~loyalty['Loyalty Number'].isin(
        cancelled.loc[cancelled['is_bad'], 'Loyalty Number']
    )
]

### Step 3: Defining the Churn Label

Defining churn is critical for a churn prediction project. In this analysis, a customer is considered to have 'churned' if they meet any of the following criteria:

*   **Formal Churn**: The customer has an explicit `Cancellation Year` recorded.
*   **Behavioral Churn**: The customer has been inactive, meaning their last flight activity occurred 6 or more months before the end of the dataset (December 2018).
*   **Never Flew**: The customer enrolled but never recorded any flight activity.

This comprehensive definition aims to capture both explicit and implicit churn behaviors.

In [ ]:
activity["activity_ym"]=activity["Year"]*100 + activity["Month"]

#### Preparing Activity Data for Churn Calculation

First, we create a combined 'YYYYMM' integer (`activity_ym`) from the `Year` and `Month` columns in the `activity` DataFrame. This standardization helps in comparing and calculating activity timelines.

In [ ]:
active_flights=activity[activity["Total Flights"]>0]

#### Filtering Active Flights

We focus only on records where `Total Flights` is greater than 0, as these represent actual customer activity. This helps in accurately determining the last active month for each customer.

In [ ]:
last_active = active_flights.groupby('Loyalty Number')['activity_ym'].max().reset_index()

#### Determining Last Active Month

For each `Loyalty Number`, we find the maximum `activity_ym` from the `active_flights` data. This indicates the most recent month a customer had any flight activity.

In [ ]:
last_active.columns = ['Loyalty Number', 'Last_Active_YM']
last_active

,Loyalty Number,Last_Active_YM
0,100018,201812
1,100102,201812
2,100140,201811
3,100214,201812
4,100272,201811
...,...,...
15162,999788,201710
15163,999902,201810
15164,999940,201812
15165,999982,201811


In [ ]:
loyalty_clean= loyalty_clean.merge(last_active, on='Loyalty Number', how='left')

#### Converting Year-Month to Total Months

To calculate time differences, we convert the 'YYYYMM' format into a continuous count of months since a base year (e.g., year * 12 + month). This makes it easier to calculate durations like 'months since last flight'.

In [ ]:
def ym_to_months(ym):
    year = ym // 100
    month = ym % 100
    return year * 12 + month

loyalty_clean['Last_Active_Months'] = loyalty_clean['Last_Active_YM'].apply(
    lambda x: ym_to_months(x) if pd.notna(x) else np.nan
)

#### Calculating Months Since Last Flight

We define the end of our dataset as December 2018 and calculate the `Months_Since_Last_Flight` for each customer. This metric is crucial for identifying behavioral churn based on inactivity.

In [ ]:
DATASET_END_YM = 201812
dataset_end_months = ym_to_months(DATASET_END_YM)
loyalty_clean['Months_Since_Last_Flight'] = dataset_end_months - loyalty_clean['Last_Active_Months']

#### Creating Churn Flags

Based on our churn definition, we create three distinct flags:

*   `Formal_Churn`: Set to 1 if a customer has a `Cancellation Year`.
*   `Behavioral_Churn`: Set to 1 if `Months_Since_Last_Flight` is 6 or more.
*   `Never_Flew`: Set to 1 if `Last_Active_YM` is null (meaning no recorded flights).

Finally, the `Churn` label is set to 1 if any of these conditions are met, capturing all forms of churn.

In [ ]:
#Formal Churn
#If customer has a cancellation year → churned officially.
loyalty_clean['Formal_Churn'] = loyalty_clean['Cancellation Year'].notna().astype(int)

# Behavioral churn flag (inactive 6+ months before end of dataset)
loyalty_clean['Behavioral_Churn'] = (loyalty_clean['Months_Since_Last_Flight'] >= 6).astype(int)

# Never flew at all
loyalty_clean['Never_Flew'] = loyalty_clean['Last_Active_YM'].isnull().astype(int)

# Final churn label
loyalty_clean['Churn'] = ((loyalty_clean['Formal_Churn'] == 1) |
                    (loyalty_clean['Behavioral_Churn'] == 1) |
                    (loyalty_clean['Never_Flew'] == 1)).astype(int)

#### Churn Breakdown and Overall Rate

We print a summary of the churn categories and the overall churn rate. This provides an initial understanding of the churn landscape within our customer base.

In [ ]:
print(f"\nChurn breakdown:")
print(f"  Formal churn     : {loyalty_clean['Formal_Churn'].sum()}")
print(f"  Behavioral churn : {loyalty_clean['Behavioral_Churn'].sum()}")
print(f"  Never flew       : {loyalty_clean['Never_Flew'].sum()}")
print(f"  Total churned    : {loyalty_clean['Churn'].sum()} ({loyalty_clean['Churn'].mean()*100:.1f}%)")
print(f"  Active           : {(loyalty_clean['Churn']==0).sum()} ({(loyalty_clean['Churn']==0).mean()*100:.1f}%)")


Churn breakdown:
  Formal churn     : 2056
  Behavioral churn : 899
  Never flew       : 1559
  Total churned    : 2783 (16.6%)
  Active           : 13943 (83.4%)


# Feature Engineering

In [ ]:
activity

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,activity_ym
0,100590,2018,6,12,15276,22914.0,0,0,201806
1,100590,2018,7,12,9168,13752.0,0,0,201807
2,100590,2018,5,4,6504,9756.0,0,0,201805
3,100590,2018,10,0,0,0.0,512,92,201810
4,100590,2018,2,0,0,0.0,0,0,201802
...,...,...,...,...,...,...,...,...,...
392931,999986,2018,4,0,0,0.0,0,0,201804
392932,999986,2018,5,0,0,0.0,0,0,201805
392933,999986,2018,6,0,0,0.0,0,0,201806
392934,999986,2018,9,0,0,0.0,0,0,201809


In [ ]:
agg = activity.groupby('Loyalty Number').agg(
    Total_Flights        = ('Total Flights', 'sum'),
    Total_Distance       = ('Distance', 'sum'),
    Total_Points_Acc     = ('Points Accumulated', 'sum'),
    Total_Points_Red     = ('Points Redeemed', 'sum'),
    Total_Dollar_Red     = ('Dollar Cost Points Redeemed', 'sum'),
    Avg_Flights_Month    = ('Total Flights', 'mean'),
    Max_Flights_Month    = ('Total Flights', 'max'),
    Active_Months        = ('Total Flights', lambda x: (x > 0).sum()),
    Total_Months         = ('Total Flights', 'count'),
).reset_index()

In [ ]:
agg["Redemption_Ratio"]=agg["Total_Points_Red"]/(agg["Total_Points_Acc"] + 0.0001)
agg["Activity_Rate"]=agg["Active_Months"]/(agg["Total_Flights"]+0.0001)
agg["Avg_Distance_Per_Flight"]=agg["Total_Distance"]/(agg["Total_Flights"]+0.0001)
agg['Quarter'] = pd.cut(activity['Month'],
                             bins=[0,3,6,9,12],
                             labels=['Q1','Q2','Q3','Q4'])


In [ ]:
#how many flights does the customer takes in each quarter?

In [ ]:
seasonal = agg[agg['Total_Flights'] > 0].groupby(
    ['Loyalty Number','Quarter']
)['Total_Flights'].sum().unstack(fill_value=0).reset_index()

seasonal.columns = [
    'Loyalty Number',
    'Flights_Q1',
    'Flights_Q2',
    'Flights_Q3',
    'Flights_Q4'
]

/tmp/ipykernel_1047/97380331.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  seasonal = agg[agg['Total_Flights'] > 0].groupby(


In [ ]:
agg=pd.merge(agg,seasonal,on="Loyalty Number")

In [ ]:
# Compare 2017 vs 2018 total flights
flights_2017 = activity[activity['Year']==2017].groupby('Loyalty Number')['Total Flights'].sum().reset_index()
flights_2017.columns = ['Loyalty Number','Flights_2017']
flights_2018 = activity[activity['Year']==2018].groupby('Loyalty Number')['Total Flights'].sum().reset_index()
flights_2018.columns = ['Loyalty Number','Flights_2018']

In [ ]:
flights_2017

,Loyalty Number,Flights_2017
0,100018,24
1,100102,25
2,100140,22
3,100214,10
4,100272,20
...,...,...
15761,999891,0
15762,999902,27
15763,999911,0
15764,999940,7


In [ ]:
flights_2018

,Loyalty Number,Flights_2018
0,100018,22
1,100102,26
2,100140,25
3,100214,12
4,100272,17
...,...,...
16732,999902,23
16733,999911,0
16734,999940,11
16735,999982,6


In [ ]:
#Trend: is the customer flying more or less over time?

In [ ]:
agg=pd.merge(agg,flights_2017,on="Loyalty Number")
agg=pd.merge(agg,flights_2018,on="Loyalty Number")
agg["Flights_Trend"]=agg["Flights_2018"]-agg["Flights_2017"]

In [ ]:
# Positive = flying more, Negative = flying less, 0 = same

In [ ]:
loyalty_clean["Tenure"]= 2018-loyalty_clean["Enrollment Year"]

In [ ]:
loyalty_clean

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,...,Salary_Imputed,enroll_ym,Last_Active_YM,Last_Active_Months,Months_Since_Last_Flight,Formal_Churn,Behavioral_Churn,Never_Flew,Churn,Tenure
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,...,83236.0,201602,201812.0,24228.0,0.0,0,0,0,0,2
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,...,73510.0,201603,201812.0,24228.0,0.0,0,0,0,0,2
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,...,73510.0,201407,201801.0,24217.0,11.0,1,1,0,1,4
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,...,73510.0,201302,201812.0,24228.0,0.0,0,0,0,0,5
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,...,103495.0,201410,201812.0,24228.0,0.0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16721,823768,Canada,British Columbia,Vancouver,V6E 3Z3,Female,College,NaN,Married,Star,...,73510.0,201212,201812.0,24228.0,0.0,0,0,0,0,6
16722,680886,Canada,Saskatchewan,Regina,S1J 3C5,Female,Bachelor,89210.0,Married,Star,...,89210.0,201409,201812.0,24228.0,0.0,0,0,0,0,4
16723,776187,Canada,British Columbia,Vancouver,V5R 1W3,Male,College,NaN,Single,Star,...,73510.0,201403,201812.0,24228.0,0.0,0,0,0,0,4
16724,906428,Canada,Yukon,Whitehorse,Y2K 6R0,Male,Bachelor,NaN,Married,Star,...,71412.0,201804,201812.0,24228.0,0.0,0,0,0,0,0


In [ ]:
loyalty_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16726 entries, 0 to 16725
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Loyalty Number            16726 non-null  int64  
 1   Country                   16726 non-null  object 
 2   Province                  16726 non-null  object 
 3   City                      16726 non-null  object 
 4   Postal Code               16726 non-null  object 
 5   Gender                    16726 non-null  object 
 6   Education                 16726 non-null  object 
 7   Salary                    12470 non-null  float64
 8   Marital Status            16726 non-null  object 
 9   Loyalty Card              16726 non-null  object 
 10  CLV                       16726 non-null  float64
 11  Enrollment Type           16726 non-null  object 
 12  Enrollment Year           16726 non-null  int64  
 13  Enrollment Month          16726 non-null  int64  
 14  Cancel

In [ ]:
#CATEGORICAL COLUMNS: Gender, Marital Status, Loyalty Card, Enrollment Type, Education

In [ ]:
#Gender Encoding
loyalty_clean["Male_Enc"]=pd.get_dummies(loyalty_clean['Gender'], drop_first=True).astype(int)

In [ ]:
#Marital Encoding
loyalty_clean['Marital_Enc'] = loyalty_clean['Marital Status'].map({
    'Single': 0,
    'Married': 1,
    'Divorced': 2
})

In [ ]:
#Education Encoding
loyalty_clean['Education_Enc'] = loyalty_clean['Education'].map({
    'High School or Below': 0,
    'College': 1,
    'Bachelor': 2,
    'Master': 3,
    'Doctor': 4
})

In [ ]:
#Loyalty Card Encoding
loyalty_clean["Card_Enc"]=loyalty_clean["Loyalty Card"].map({
 'Star': 0,
 'Nova': 1,
 'Aurora': 2
})

In [ ]:
#Promo Encoding
loyalty_clean['Promo_Enroll']  = (loyalty_clean['Enrollment Type'] == '2018 Promotion').astype(int)

In [ ]:
model_df = pd.merge(loyalty_clean, agg, on='Loyalty Number')

In [ ]:
#replacing null values with 0

model_df.fillna({
    'Total_Flights': 0,
    'Total_Distance': 0,
    'Total_Points_Acc': 0,
    'Total_Points_Red': 0,
    'Active_Months': 0,
    'Activity_Rate': 0,
    'Redemption_Ratio': 0,
    'Flight_Trend': 0,
    'Flights_Q1': 0, 'Flights_Q2': 0, 'Flights_Q3': 0, 'Flights_Q4': 0
}, inplace=True)

In [ ]:
features = [
    'Loyalty Number',
    # Demographics
    'Male_Enc', 'Education_Enc', 'Marital_Enc', 'Card_Enc',
    'Salary_Imputed', 'Salary_Was_Imputed', 'CLV', 'Tenure', 'Promo_Enroll',
    # Behavioral
    'Total_Flights', 'Total_Distance', 'Total_Points_Acc', 'Total_Points_Red',
    'Avg_Flights_Month', 'Max_Flights_Month', 'Active_Months', 'Activity_Rate',
    'Redemption_Ratio', 'Avg_Distance_Per_Flight', 'Months_Since_Last_Flight',
    # Seasonal
    'Flights_Q1', 'Flights_Q2', 'Flights_Q3', 'Flights_Q4',
    # Trend
    'Flights_Trend', 'Flights_2017', 'Flights_2018',
    # Label
    'Churn', 'Formal_Churn', 'Behavioral_Churn', 'Never_Flew'
]

model_df = model_df[features]

In [ ]:
print(f"\nFinal modeling dataset shape: {model_df.shape}")
print(f"Features: {len(features)-4} (excluding ID + 3 churn breakdown cols)")
print(f"\nNull check:\n{model_df.isnull().sum()[model_df.isnull().sum() > 0]}")
print(f"\nSample:\n{model_df.head(3).to_string()}")



Final modeling dataset shape: (14243, 32)
Features: 28 (excluding ID + 3 churn breakdown cols)

Null check:
Series([], dtype: int64)

Sample:
   Loyalty Number  Male_Enc  Education_Enc  Marital_Enc  Card_Enc  Salary_Imputed  Salary_Was_Imputed      CLV  Tenure  Promo_Enroll  Total_Flights  Total_Distance  Total_Points_Acc  Total_Points_Red  Avg_Flights_Month  Max_Flights_Month  Active_Months  Activity_Rate  Redemption_Ratio  Avg_Distance_Per_Flight  Months_Since_Last_Flight  Flights_Q1  Flights_Q2  Flights_Q3  Flights_Q4  Flights_Trend  Flights_2017  Flights_2018  Churn  Formal_Churn  Behavioral_Churn  Never_Flew
0          480934         0              2            1         0         83236.0               False  3839.14       2             0             37           54525           54525.0              1418           1.541667                  5             14       0.378377          0.026006              1473.644666                       0.0           0           0           0      

In [1]:
model_df.to_csv("Modeling_Dataset.csv", index=False)
print("\n Saved as Modeling_Dataset.csv")

NameError: name 'model_df' is not defined

In [ ]:
import pandas as pd

df = pd.read_csv("Modeling_Dataset.csv")

print("\nChurn distribution:")
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")



Churn distribution:
Churn
0    13134
1     1109
Name: count, dtype: int64
Churn rate: 7.8%
